# Freedom Score — ML-based User Value Score

This notebook is aligned with the hackathon criterion:

> **ML-based User Value Score (Freedom Score)**  
> Develop an ML model that aggregates user behavior, transactions, and product usage into a single score reflecting user value and potential.

Expected outputs covered here:

- **Scoring model**: LightGBM binary classification model predicting high-value users.
- **Feature importance**: model importance + SHAP-ready block for business interpretation.
- **Segmentation**: High / Medium / Low value users based on the score.

Important data fix vs the previous version:

- The raw file has repeated `customer_id` values.
- A row-level train/test split leaks the same customer into train and test.
- This version first builds a **one-row-per-customer** modeling table, then validates with customer-level splits.

## 0. Imports and Configuration

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)
from sklearn.inspection import permutation_importance

import lightgbm as lgb

try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False

RANDOM_STATE = 42

# Change this path if the CSV is moved next to the notebook.
DATA_PATH = Path(r"C:\Users\Professional\Downloads\master_features.csv")
OUTPUT_DIR = Path(".")

print("Ready")
print(f"Data path: {DATA_PATH}")

Ready
Data path: C:\Users\Professional\Downloads\master_features.csv


## 1. Load Data and Validate Granularity

In [2]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Raw rows: {len(df_raw):,}")
print(f"Raw columns: {df_raw.shape[1]:,}")
print(f"Unique customer_id: {df_raw['customer_id'].nunique():,}")
print(f"Duplicated customer_id rows: {df_raw['customer_id'].duplicated().sum():,}")

dupe_stats = (
    df_raw.groupby("customer_id")
    .agg(
        rows=("customer_id", "size"),
        city_n=("city", "nunique"),
        gender_n=("gender", "nunique"),
        age_n=("customer_age", "nunique"),
        reg_date_n=("reg_date", "nunique"),
        target_n=("is_superapp_adopter", "nunique"),
    )
)

print("\nRows per customer:")
print(dupe_stats["rows"].describe(percentiles=[0.5, 0.9, 0.99]).to_string())

print("\nShare of customer_ids with conflicting profile fields:")
for col in ["city_n", "gender_n", "age_n", "reg_date_n", "target_n"]:
    print(f"  {col:10s}: {(dupe_stats[col].gt(1).mean() * 100):5.2f}%")

Raw rows: 899,188
Raw columns: 50
Unique customer_id: 475,216
Duplicated customer_id rows: 423,972



Rows per customer:
count    475216.000000
mean          1.892167
std           0.800658
min           1.000000
50%           2.000000
90%           3.000000
99%           3.000000
max           3.000000

Share of customer_ids with conflicting profile fields:
  city_n    : 57.03%
  gender_n  : 36.08%
  age_n     : 61.04%
  reg_date_n: 12.03%
  target_n  :  0.00%


## 2. Build One Row per Customer

The raw file contains repeated `customer_id`. For a customer-level score, the modeling table must have one row per customer.

Aggregation choices:

- Numeric behavior fields: mean, sum, max, or median depending on business meaning.
- Profile fields: mode / first stable value.
- Target-style ecosystem usage: max, because any partner usage means the user has adopted.

This keeps validation honest and makes the exported score one row per customer.

In [3]:
def mode_or_first(s):
    s = s.dropna()
    if s.empty:
        return np.nan
    mode = s.mode()
    return mode.iloc[0] if len(mode) else s.iloc[0]


sum_cols = [
    "txn_count", "txn_total_spend", "proc_total_attempts", "proc_completed_count",
    "pp_purchase_count", "pp_total_spend", "pp_total_cashback",
]

max_cols = [
    "txn_max_spend", "txn_unique_mcc", "txn_unique_terminals", "proc_unique_processes",
    "proc_langs_used", "pp_unique_apps", "engage_has_txn", "engage_has_pp",
    "engage_has_proc", "is_superapp_adopter",
]

min_cols = ["txn_recency_days", "pp_recency_days"]

mean_cols = [
    "txn_avg_spend", "txn_median_spend", "txn_std_spend", "txn_freq_per_day",
    "op_share_atm_cash_withdrawal", "op_share_cash-in", "op_share_p2p_credit",
    "op_share_p2p_debit", "op_share_pos_cash_advance", "op_share_payment_transaction",
    "op_share_purchase", "op_share_purchase_return_(credit)",
    "op_share_transaction_cost_inquiry", "proc_avg_duration_sec",
    "proc_completion_rate", "pp_avg_spend", "pp_avg_cashback_rate",
]

profile_cols = ["city", "gender", "reg_date", "acq_channel", "age_group", "pp_top_app"]

agg = {}
for c in sum_cols:
    if c in df_raw.columns:
        agg[c] = "sum"
for c in max_cols:
    if c in df_raw.columns:
        agg[c] = "max"
for c in min_cols:
    if c in df_raw.columns:
        agg[c] = "min"
for c in mean_cols:
    if c in df_raw.columns:
        agg[c] = "mean"
for c in profile_cols:
    if c in df_raw.columns:
        agg[c] = mode_or_first

if "customer_age" in df_raw.columns:
    agg["customer_age"] = "median"
if "account_age_days" in df_raw.columns:
    agg["account_age_days"] = "max"
if "txn_tenure_days" in df_raw.columns:
    agg["txn_tenure_days"] = "max"
if "engagement_score" in df_raw.columns:
    agg["engagement_score"] = "max"

df = df_raw.groupby("customer_id", as_index=False).agg(agg)

print(f"Customer-level rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Target adoption rate: {df['is_superapp_adopter'].mean() * 100:.2f}%")

Customer-level rows: 475,216
Columns: 47
Target adoption rate: 20.48%


## 3. Create a Transparent Value Proxy

There is no explicit labeled revenue/LTV target in the file, so we define a transparent proxy for **user value and potential**.

The proxy combines:

- transaction value: total spend and transaction count;
- app/product engagement: process attempts, completed processes, breadth of processes;
- ecosystem usage: partner purchases and superapp adoption;
- recency/potential: recent activity and account age.

Then we train a model to predict the top-value users. This gives an ML score that can be applied to new users with the same feature set.

In [4]:
df_feat = df.copy()

# Activity nulls mean no recorded activity in that domain.
txn_cols = [c for c in df_feat.columns if c.startswith("txn_") or c.startswith("op_share_")]
pp_cols = [c for c in df_feat.columns if c.startswith("pp_") and c != "pp_top_app"]
flag_cols = [c for c in ["engage_has_txn", "engage_has_pp", "engage_has_proc", "is_superapp_adopter"] if c in df_feat.columns]

df_feat[txn_cols] = df_feat[txn_cols].fillna(0)
df_feat[pp_cols] = df_feat[pp_cols].fillna(0)
df_feat[flag_cols] = df_feat[flag_cols].fillna(0)

proc_cols = [c for c in df_feat.columns if c.startswith("proc_")]
for c in proc_cols:
    df_feat[c] = df_feat[c].fillna(df_feat[c].median())

df_feat["account_age_days"] = df_feat["account_age_days"].clip(lower=1)
df_feat["spend_cv"] = df_feat["txn_std_spend"] / (df_feat["txn_avg_spend"] + 1)
df_feat["txn_intensity"] = df_feat["txn_count"] / df_feat["account_age_days"]
df_feat["purchase_dominance"] = df_feat["op_share_purchase"] / (
    df_feat["op_share_purchase"] + df_feat["op_share_atm_cash_withdrawal"] + 1e-6
)
df_feat["app_engagement_depth"] = df_feat["proc_completion_rate"] * df_feat["proc_unique_processes"]
df_feat["ecosystem_breadth"] = df_feat.get("pp_unique_apps", 0) + df_feat.get("is_superapp_adopter", 0)
df_feat["recent_txn_signal"] = 1 / (1 + df_feat["txn_recency_days"])
df_feat["acquired_via_partner"] = df_feat["acq_channel"].isin(["arbuz", "ticketon", "ftravel"]).astype(int)

for c in [
    "spend_cv", "txn_intensity", "purchase_dominance", "app_engagement_depth",
    "ecosystem_breadth", "recent_txn_signal",
]:
    df_feat[c] = df_feat[c].replace([np.inf, -np.inf], np.nan).fillna(0)

def robust_minmax(s, low=0.01, high=0.99):
    lo, hi = s.quantile(low), s.quantile(high)
    return ((s.clip(lo, hi) - lo) / (hi - lo + 1e-9)).clip(0, 1)

value_components = pd.DataFrame(index=df_feat.index)
value_components["spend"] = robust_minmax(np.log1p(df_feat["txn_total_spend"]))
value_components["frequency"] = robust_minmax(np.log1p(df_feat["txn_count"]))
value_components["app_depth"] = robust_minmax(df_feat["app_engagement_depth"])
value_components["product_breadth"] = robust_minmax(df_feat["proc_unique_processes"])
value_components["ecosystem"] = robust_minmax(
    np.log1p(df_feat.get("pp_purchase_count", 0)) + df_feat["ecosystem_breadth"]
)
value_components["recency"] = robust_minmax(df_feat["recent_txn_signal"])

weights = {
    "spend": 0.25,
    "frequency": 0.20,
    "app_depth": 0.20,
    "product_breadth": 0.15,
    "ecosystem": 0.15,
    "recency": 0.05,
}

df_feat["value_proxy"] = sum(value_components[k] * w for k, w in weights.items())

# Classification target: top 30% by transparent value proxy.
threshold = df_feat["value_proxy"].quantile(0.70)
df_feat["high_value_target"] = (df_feat["value_proxy"] >= threshold).astype(int)

print(f"High-value threshold: {threshold:.4f}")
print(f"High-value users: {df_feat['high_value_target'].mean() * 100:.1f}%")
print("\nValue proxy components:")
print(value_components.describe().T[["mean", "std", "min", "max"]].round(3).to_string())

High-value threshold: 0.4526
High-value users: 30.0%

Value proxy components:
                  mean    std  min  max
spend            0.380  0.409  0.0  1.0
frequency        0.207  0.267  0.0  1.0
app_depth        0.404  0.194  0.0  1.0
product_breadth  0.403  0.202  0.0  1.0
ecosystem        0.125  0.258  0.0  1.0
recency          0.619  0.459  0.0  1.0


## 4. Feature Set

The model uses behavior, transactions, app/product usage, and acquisition signals.

To keep feature importance meaningful, it does **not** use `value_proxy` or the final target as model inputs.

In [5]:
FEATURE_COLS = [
    # profile
    "customer_age",
    "account_age_days",

    # transaction behavior
    "txn_count",
    "txn_total_spend",
    "txn_avg_spend",
    "txn_median_spend",
    "txn_std_spend",
    "txn_max_spend",
    "txn_unique_mcc",
    "txn_unique_terminals",
    "txn_recency_days",
    "txn_tenure_days",
    "txn_freq_per_day",
    "op_share_purchase",
    "op_share_p2p_credit",
    "op_share_p2p_debit",
    "op_share_atm_cash_withdrawal",
    "op_share_payment_transaction",

    # product / app usage
    "proc_total_attempts",
    "proc_completed_count",
    "proc_completion_rate",
    "proc_unique_processes",
    "proc_avg_duration_sec",
    "proc_langs_used",
    "engage_has_txn",
    "engage_has_proc",

    # ecosystem usage as current value signal
    "is_superapp_adopter",
    "pp_purchase_count",
    "pp_total_spend",
    "pp_total_cashback",
    "pp_unique_apps",

    # engineered features
    "spend_cv",
    "txn_intensity",
    "purchase_dominance",
    "app_engagement_depth",
    "ecosystem_breadth",
    "recent_txn_signal",
    "acquired_via_partner",
]

FEATURE_COLS = [c for c in FEATURE_COLS if c in df_feat.columns]
TARGET_COL = "high_value_target"

X = df_feat[FEATURE_COLS].copy()
y = df_feat[TARGET_COL].copy()

imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=FEATURE_COLS, index=df_feat.index)

print(f"Features used: {len(FEATURE_COLS)}")
print(FEATURE_COLS)

Features used: 38
['customer_age', 'account_age_days', 'txn_count', 'txn_total_spend', 'txn_avg_spend', 'txn_median_spend', 'txn_std_spend', 'txn_max_spend', 'txn_unique_mcc', 'txn_unique_terminals', 'txn_recency_days', 'txn_tenure_days', 'txn_freq_per_day', 'op_share_purchase', 'op_share_p2p_credit', 'op_share_p2p_debit', 'op_share_atm_cash_withdrawal', 'op_share_payment_transaction', 'proc_total_attempts', 'proc_completed_count', 'proc_completion_rate', 'proc_unique_processes', 'proc_avg_duration_sec', 'proc_langs_used', 'engage_has_txn', 'engage_has_proc', 'is_superapp_adopter', 'pp_purchase_count', 'pp_total_spend', 'pp_total_cashback', 'pp_unique_apps', 'spend_cv', 'txn_intensity', 'purchase_dominance', 'app_engagement_depth', 'ecosystem_breadth', 'recent_txn_signal', 'acquired_via_partner']


## 5. Customer-Level Train / Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imp,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = neg / pos

print(f"Training rows: {len(X_train):,} | high-value rate: {y_train.mean() * 100:.1f}%")
print(f"Test rows:     {len(X_test):,} | high-value rate: {y_test.mean() * 100:.1f}%")
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

Training rows: 380,172 | high-value rate: 30.0%
Test rows:     95,044 | high-value rate: 30.0%
scale_pos_weight: 2.333


## 6. Train Scoring Model

In [7]:
model = lgb.LGBMClassifier(
    objective="binary",
    metric="auc",
    n_estimators=600,
    learning_rate=0.04,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=80,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100),
    ],
)

print(f"Best iteration: {model.best_iteration_}")

[100]	valid_0's auc: 0.999921


[200]	valid_0's auc: 0.999972


[300]	valid_0's auc: 0.999981


[400]	valid_0's auc: 0.999983


[500]	valid_0's auc: 0.999984


[600]	valid_0's auc: 0.999985
Best iteration: 591


## 7. Model Evaluation

In [8]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
report = classification_report(y_test, y_pred, output_dict=True)

print(f"AUC-ROC:           {auc:.4f}")
print(f"Average Precision: {ap:.4f}")
print(f"Accuracy:          {report['accuracy']:.4f}")
print(f"Precision high:    {report['1']['precision']:.4f}")
print(f"Recall high:       {report['1']['recall']:.4f}")
print(f"F1 high:           {report['1']['f1-score']:.4f}")
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

AUC-ROC:           1.0000
Average Precision: 1.0000
Accuracy:          0.9981
Precision high:    0.9954
Recall high:       0.9981
F1 high:           0.9968

Confusion matrix:
[[66400   131]
 [   54 28459]]


## 8. Cross-Validation Stability

In [9]:
cv_params = model.get_params()
cv_params["n_estimators"] = model.best_iteration_ or 300
cv_model = lgb.LGBMClassifier(**cv_params)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(cv_model, X_imp, y, cv=cv, scoring="roc_auc", n_jobs=-1)

print("5-fold AUC scores:")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"Mean AUC: {cv_scores.mean():.4f}")
print(f"Std AUC:  {cv_scores.std():.4f}")

5-fold AUC scores:
  Fold 1: 1.0000
  Fold 2: 1.0000
  Fold 3: 1.0000
  Fold 4: 1.0000
  Fold 5: 1.0000
Mean AUC: 1.0000
Std AUC:  0.0000


## 9. Feature Importance — What Drives Value

In [10]:
importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "gain_importance": model.booster_.feature_importance(importance_type="gain"),
    "split_importance": model.booster_.feature_importance(importance_type="split"),
}).sort_values("gain_importance", ascending=False)

importance_df["gain_share"] = importance_df["gain_importance"] / importance_df["gain_importance"].sum()

print("Top 20 drivers of high user value:")
print(importance_df.head(20).to_string(index=False))

Top 20 drivers of high user value:
              feature  gain_importance  split_importance  gain_share
            txn_count     4.780875e+06              4046    0.528496
      txn_total_spend     1.643351e+06              4815    0.181662
 app_engagement_depth     8.460152e+05              5909    0.093522
  is_superapp_adopter     3.822356e+05               785    0.042254
proc_unique_processes     3.746827e+05              3260    0.041419
    pp_purchase_count     2.121373e+05              1059    0.023450
        txn_intensity     1.770232e+05               960    0.019569
     txn_recency_days     1.595478e+05              2970    0.017637
       pp_total_spend     1.551187e+05               888    0.017147
        txn_max_spend     1.091006e+05               802    0.012060
 proc_completion_rate     5.162128e+04              1911    0.005706
    pp_total_cashback     3.713751e+04               793    0.004105
        txn_avg_spend     2.971277e+04               847    0.003285

In [11]:
# Optional permutation importance: slower, but easier to explain as model performance drop.
sample_n = min(25_000, len(X_test))
X_perm = X_test.sample(sample_n, random_state=RANDOM_STATE)
y_perm = y_test.loc[X_perm.index]

perm = permutation_importance(
    model,
    X_perm,
    y_perm,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="roc_auc",
    n_jobs=-1,
)

perm_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "auc_drop_mean": perm.importances_mean,
    "auc_drop_std": perm.importances_std,
}).sort_values("auc_drop_mean", ascending=False)

print("Top 15 permutation importance features:")
print(perm_df.head(15).to_string(index=False))

Top 15 permutation importance features:
              feature  auc_drop_mean  auc_drop_std
            txn_count       0.015818  4.353573e-04
 app_engagement_depth       0.011359  3.889538e-04
      txn_total_spend       0.010245  3.087340e-04
proc_unique_processes       0.004436  1.747443e-04
     txn_recency_days       0.001511  7.690529e-05
  is_superapp_adopter       0.001117  2.986838e-05
    pp_purchase_count       0.000436  2.538968e-05
       pp_total_spend       0.000030  4.171795e-06
       pp_unique_apps       0.000021  8.859717e-07
    recent_txn_signal       0.000018  2.034221e-06
        txn_intensity       0.000013  8.384399e-07
    pp_total_cashback       0.000013  1.650311e-06
        txn_max_spend       0.000010  8.849530e-07
        txn_avg_spend       0.000002  5.979619e-07
 proc_completion_rate       0.000002  6.194954e-07


In [12]:
if SHAP_AVAILABLE:
    sample_n = min(5_000, len(X_test))
    X_shap = X_test.sample(sample_n, random_state=RANDOM_STATE)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    shap_importance = pd.DataFrame({
        "feature": FEATURE_COLS,
        "mean_abs_shap": np.abs(sv).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False)

    print("Top 20 SHAP features:")
    print(shap_importance.head(20).to_string(index=False))
else:
    print("SHAP is not installed in this environment. Use gain/permutation importance above.")

SHAP is not installed in this environment. Use gain/permutation importance above.


## 10. Freedom Score and Segmentation

The model output is the predicted probability that a customer belongs to the high-value group.

We convert that probability to a 0-100 **Freedom Score**:

```text
Freedom Score = P(high-value user) * 100
```

Segments:

- **High**: top 30% by score
- **Medium**: middle 40%
- **Low**: bottom 30%

In [13]:
df_feat["high_value_probability"] = model.predict_proba(X_imp)[:, 1]
df_feat["freedom_score"] = (df_feat["high_value_probability"] * 100).round(1)

p30 = df_feat["freedom_score"].quantile(0.30)
p70 = df_feat["freedom_score"].quantile(0.70)

df_feat["value_segment"] = pd.cut(
    df_feat["freedom_score"],
    bins=[-0.001, p30, p70, 100.001],
    labels=["Low", "Medium", "High"],
)

print(f"Low / Medium threshold:  {p30:.1f}")
print(f"Medium / High threshold: {p70:.1f}")
print("\nSegment sizes:")
print(df_feat["value_segment"].value_counts(normalize=True).mul(100).round(1).to_string())

Low / Medium threshold:  0.0
Medium / High threshold: 61.5

Segment sizes:
value_segment
Low       66.9
High      30.0
Medium     3.1


## 11. Segment Profiles and Business Actions

In [14]:
profile_cols = [
    "freedom_score",
    "value_proxy",
    "txn_count",
    "txn_total_spend",
    "txn_unique_mcc",
    "proc_total_attempts",
    "proc_completion_rate",
    "proc_unique_processes",
    "pp_purchase_count",
    "pp_total_spend",
    "is_superapp_adopter",
]
profile_cols = [c for c in profile_cols if c in df_feat.columns]

segment_profile = (
    df_feat.groupby("value_segment", observed=True)[profile_cols]
    .mean()
    .round(3)
)

segment_sizes = df_feat["value_segment"].value_counts().rename("users")

print("Segment sizes:")
print(segment_sizes.to_string())
print("\nSegment profiles:")
print(segment_profile.to_string())

Segment sizes:
value_segment
Low       317839
High      142565
Medium     14812

Segment profiles:
               freedom_score  value_proxy  txn_count  txn_total_spend  txn_unique_mcc  proc_total_attempts  proc_completion_rate  proc_unique_processes  pp_purchase_count  pp_total_spend  is_superapp_adopter
value_segment                                                                                                                                                                                                  
Low                    0.000        0.202      0.688        65170.577           0.243               25.878                 0.836                  5.088              0.068        5958.486                0.036
Medium                 4.819        0.438      5.290       597515.232           1.256               38.047                 0.855                  6.615              0.979       81366.075                0.324
High                  99.766        0.596     30.797      3760513.385

Business interpretation:

| Segment | Meaning | Recommended action |
|---|---|---|
| **High** | Highest modeled value and strongest product usage | Retention, premium cashback, loyalty, cross-sell to ecosystem products |
| **Medium** | Good potential but not fully monetized | Personalized offers, onboarding into underused products, partner promos |
| **Low** | Low activity or weak engagement | Basic activation, education, low-friction first transaction nudges |

## 12. Save Model Bundle and Scored Users

In [15]:
model_bundle = {
    "model": model,
    "imputer": imputer,
    "feature_cols": FEATURE_COLS,
    "target_col": TARGET_COL,
    "score_definition": "Freedom Score = predicted probability of high-value user * 100",
    "value_proxy_weights": weights,
    "segment_thresholds": {"low_medium": float(p30), "medium_high": float(p70)},
    "metrics": {
        "test_auc": float(auc),
        "test_average_precision": float(ap),
        "cv_auc_mean": float(cv_scores.mean()),
        "cv_auc_std": float(cv_scores.std()),
    },
}

joblib.dump(model_bundle, OUTPUT_DIR / "freedom_score_value_model.pkl")

export_cols = [
    "customer_id", "city", "gender", "customer_age", "account_age_days",
    "freedom_score", "high_value_probability", "value_segment", "value_proxy",
    "txn_count", "txn_total_spend", "txn_unique_mcc",
    "proc_completion_rate", "proc_unique_processes",
    "pp_purchase_count", "pp_total_spend", "is_superapp_adopter",
]
export_cols = [c for c in export_cols if c in df_feat.columns]

export_df = df_feat[export_cols].copy()
export_df["high_value_probability"] = export_df["high_value_probability"].round(4)
export_df.to_csv(OUTPUT_DIR / "freedom_scored_users_value.csv", index=False)

importance_df.to_csv(OUTPUT_DIR / "freedom_score_feature_importance.csv", index=False)

print("Saved:")
print("  freedom_score_value_model.pkl")
print("  freedom_scored_users_value.csv")
print("  freedom_score_feature_importance.csv")

Saved:
  freedom_score_value_model.pkl
  freedom_scored_users_value.csv
  freedom_score_feature_importance.csv
